# African Proverb Explainer & Illustrator

A multi-modal AI assistant that explores the wisdom of African proverbs.

Enter a proverb, ask about a specific culture, or hit **Surprise Me** to discover a random gem.
The assistant will:
- **Explain** the proverb's meaning and cultural context
- **Draw** an illustration inspired by its lesson (DALL-E)
- **Narrate** a spoken summary (TTS)

**Week 2 concepts used:** Multi-model APIs (Day 1) · Gradio UI (Day 2) · Chatbot patterns (Day 3) · Tool calling (Day 4) · Multi-modal output (Day 5)

In [ ]:
import os
import json
import random
import base64
from io import BytesIO
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
import gradio as gr

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

## Proverb Bank

A curated collection of proverbs spanning nine cultures across the continent.
The LLM uses tool calling to look these up, then crafts its explanation around the result.

In [ ]:
PROVERBS = {
    "yoruba": [
        {
            "text": "Àgbà kì í wà lójà, kí orí ọmọ títúntùn wó.",
            "translation": "An elder does not stay in the market and let a child's head go crooked.",
            "context": "Experienced people have a duty to guide the young.",
        },
        {
            "text": "Bí a bá ń lọ, a ń bọ̀.",
            "translation": "If we keep going, we are coming back.",
            "context": "Life moves in cycles; what goes around comes around.",
        },
        {
            "text": "Ọwọ́ kan ò gbé ẹrù d'órí.",
            "translation": "One hand cannot lift a load onto the head.",
            "context": "Cooperation is essential for difficult tasks.",
        },
    ],
    "igbo": [
        {
            "text": "Onye aghala nwanne ya.",
            "translation": "Be your brother's keeper.",
            "context": "Community responsibility and solidarity.",
        },
        {
            "text": "Otu onye tuo izu, o gbue ochu.",
            "translation": "When one person thinks alone, it leads to poor decisions.",
            "context": "Collective wisdom prevents costly mistakes.",
        },
        {
            "text": "Egbe bere ugo bere, nke sị ibe ya ebena, nku kwa ya.",
            "translation": "Let the kite perch, let the eagle perch; whichever denies the other, may its wing break.",
            "context": "A call for tolerance and coexistence.",
        },
    ],
    "zulu": [
        {
            "text": "Umuntu ngumuntu ngabantu.",
            "translation": "A person is a person through other people.",
            "context": "The Ubuntu philosophy — our humanity is tied to how we treat others.",
        },
        {
            "text": "Indlela ibuzwa kwabaphambili.",
            "translation": "The road is asked from those who have traveled it.",
            "context": "Seek guidance from those with experience.",
        },
    ],
    "swahili": [
        {
            "text": "Haraka haraka haina baraka.",
            "translation": "Hurry hurry has no blessing.",
            "context": "Patience leads to better outcomes than rushing.",
        },
        {
            "text": "Mgeni siku mbili; siku ya tatu mpe jembe.",
            "translation": "A guest for two days; on the third day, give them a hoe.",
            "context": "Hospitality is generous but has limits — everyone must contribute.",
        },
        {
            "text": "Mti haukui kwa siku moja.",
            "translation": "A tree does not grow in a day.",
            "context": "Great achievements require sustained effort and patience.",
        },
    ],
    "akan": [
        {
            "text": "Sɛ wo were fi na wosankofa a yennki.",
            "translation": "It is not wrong to go back for what you forgot.",
            "context": "The Sankofa principle — learn from the past to move forward.",
        },
        {
            "text": "Obi nkyerɛ abɔfra Nyame.",
            "translation": "Nobody teaches a child about God.",
            "context": "Some truths are innate and self-evident.",
        },
    ],
    "hausa": [
        {
            "text": "Ruwan sama ba ya tsayawa a kan tudu.",
            "translation": "Rainwater does not stay on a hilltop.",
            "context": "Wealth and resources flow away from those who hoard them.",
        },
        {
            "text": "Hankali ya fi karfi.",
            "translation": "Wisdom is greater than strength.",
            "context": "Intellect and careful thought outweigh brute force.",
        },
    ],
    "amhara": [
        {
            "text": "\u1230\u12CD \u1260\u1230\u12CD \u12ED\u12A8\u1260\u122B\u120D\u1362",
            "translation": "A person is honored through other people.",
            "context": "Dignity and respect come from community, not isolation.",
        },
        {
            "text": "\u12A8\u121E\u129D \u130B\u122D \u12A0\u1275\u1218\u12AB\u12A8\u122D\u1363 \u12A8\u12A0\u132D\u122D \u130B\u122D \u12A0\u1275\u12D8\u120D\u120D\u1362",
            "translation": "Don\u2019t consult with a fool, and don\u2019t jump with a short person.",
            "context": "Choose your advisors and companions wisely.",
        },
    ],
    "wolof": [
        {
            "text": "Ku am nit, am na alal.",
            "translation": "He who has people, has wealth.",
            "context": "True riches are found in relationships, not material goods.",
        },
    ],
    "maasai": [
        {
            "text": "Meeta enkop naomon.",
            "translation": "The earth is not owned.",
            "context": "We are stewards of the land, not its masters.",
        },
    ],
}

print(f"Loaded {sum(len(v) for v in PROVERBS.values())} proverbs across {len(PROVERBS)} cultures")

In [ ]:
def get_random_proverb():
    """Return a random proverb from any culture in the bank."""
    culture = random.choice(list(PROVERBS.keys()))
    proverb = random.choice(PROVERBS[culture])
    print(f"TOOL CALLED: random proverb from {culture}", flush=True)
    return json.dumps({"culture": culture, **proverb})


def get_proverbs_by_culture(culture):
    """Return proverbs for a given culture, or list available cultures."""
    key = culture.strip().lower()
    if key in PROVERBS:
        print(f"TOOL CALLED: proverbs for {key}", flush=True)
        return json.dumps({"culture": key, "proverbs": PROVERBS[key]})
    available = ", ".join(sorted(PROVERBS.keys()))
    return json.dumps({"error": f"No proverbs found for '{culture}'. Available cultures: {available}"})


random_proverb_schema = {
    "name": "get_random_proverb",
    "description": "Pick a random African proverb from the collection. Call this when the user wants a surprise or does not specify a culture.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
}

culture_proverb_schema = {
    "name": "get_proverbs_by_culture",
    "description": "Get African proverbs from a specific culture or ethnic group (e.g. yoruba, igbo, zulu, swahili, akan, hausa, amhara, wolof, maasai).",
    "parameters": {
        "type": "object",
        "properties": {
            "culture": {
                "type": "string",
                "description": "The culture or ethnic group to look up.",
            },
        },
        "required": ["culture"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": random_proverb_schema},
    {"type": "function", "function": culture_proverb_schema},
]

## System Prompt & Agent Logic

The assistant plays the role of a warm, knowledgeable cultural scholar
who explains proverbs with depth, a modern lens, and a touch of storytelling.

In [ ]:
system_message = """You are a warm, knowledgeable African cultural scholar.

When given a proverb, follow this structure:

1. State the proverb in its original language and the English translation.
2. Name the culture/ethnic group it comes from and briefly note the region.
3. Explain its meaning in 2-3 sentences.
4. Give a modern-day analogy or scenario where this wisdom applies.
5. End with a single-sentence reflection connecting it to a universal human truth.

Keep the tone conversational and engaging — like a wise elder telling stories by a fire.
Use the available tools to look up proverbs when the user asks for one.
If the user says 'surprise me' or similar, call get_random_proverb.
If they mention a specific culture, call get_proverbs_by_culture and pick one to explain."""

In [ ]:
def handle_tool_calls(message):
    """Execute every tool call the model requested and return results."""
    responses = []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        if name == "get_random_proverb":
            result = get_random_proverb()
        elif name == "get_proverbs_by_culture":
            result = get_proverbs_by_culture(args["culture"])
        else:
            result = json.dumps({"error": f"Unknown tool: {name}"})
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id,
        })
    return responses


def artist(proverb_text):
    """Generate a DALL-E illustration inspired by the proverb."""
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=(
            f"A warm, vibrant illustration inspired by this African proverb: "
            f"'{proverb_text}'. "
            f"Style: rich earthy tones, African patterns and textures, "
            f"storytelling scene, no text in the image."
        ),
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))


def talker(text):
    """Generate TTS narration of the given text."""
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=text,
    )
    return response.content

In [ ]:
def chat(history):
    """Agent loop: tool calls -> explanation -> illustration -> narration."""
    messages = [{"role": "system", "content": system_message}]
    messages += [{"role": h["role"], "content": h["content"]} for h in history]

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_results = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(tool_results)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})

    voice = talker(reply)

    first_line = reply.split("\n")[0]
    image = artist(first_line)

    return history, voice, image

## Gradio Interface

A `gr.Blocks` layout with a chatbot panel, illustration panel, and audio narration.

In [ ]:
def add_user_message(message, history):
    """Move user message into chatbot history and clear the textbox."""
    return "", history + [{"role": "user", "content": message}]


def surprise_me(history):
    """Inject a 'surprise me' request into the chat."""
    return history + [{"role": "user", "content": "Surprise me with a random African proverb!"}]


with gr.Blocks(title="African Proverb Explainer") as ui:
    gr.Markdown("# African Proverb Explainer & Illustrator")
    gr.Markdown(
        "Enter a proverb, ask about a culture (e.g. *'Tell me a Yoruba proverb'*), "
        "or click **Surprise Me** to discover a random gem."
    )
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, label="Illustration", interactive=False)
    with gr.Row():
        audio_output = gr.Audio(label="Narration", autoplay=True)
    with gr.Row():
        message = gr.Textbox(
            label="Ask about an African proverb:",
            placeholder="e.g. Tell me a Zulu proverb, or paste one you've heard...",
        )
        surprise_btn = gr.Button("Surprise Me")

    message.submit(add_user_message, [message, chatbot], [message, chatbot]).then(
        chat, chatbot, [chatbot, audio_output, image_output]
    )
    surprise_btn.click(surprise_me, chatbot, chatbot).then(
        chat, chatbot, [chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)